# 05. D3 loading, class-by-condition counts and window selection (E8)

- Signals: 280 (accelerometer 80, dynamic pressure 80, hydrophone 120); background files excluded
- Accuracy analyses: hydrophone 120 only
- Target: leak (CC, GL, LC, OL) vs. no-leak (NL); majority rate 0.800
- Operating condition = background flow × background noise (6 levels)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit

M3 = io_utils.scan_d3()
assert len(M3) > 0, "D3 empty - run notebook 00 diagnostics first."
D3all = M3[(~M3.is_bg) & M3.label.notna()].reset_index(drop=True)
print(f"D3 files: {len(M3)}   (background {int(M3.is_bg.sum())}, analysable {len(D3all)})")
print(D3all.groupby(["modality", "label"]).size().unstack().to_string())

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000
D3 files: 282   (background 2, analysable 280)
label       leak  noleak
modality                
accel         64      16
hydrophone    96      24
pressure      64      16


## 1. Populations and class-by-condition counts

- Populations: all modalities (280), hydrophone recordings (120), 1 s segments (3,600)
- P(leak) by topology, flow, noise, sensor, modality and their combinations

In [2]:
H = D3all[D3all.modality == "hydrophone"].reset_index(drop=True)
mapping = pd.DataFrame([
    dict(population="all modalities (v1 Table 3 SFB)", n=len(D3all),
         composition="; ".join(f"{k}={v}" for k, v in D3all.modality.value_counts().items()),
         n_runs=D3all.drop_duplicates(["topology", "leaktype", "flow", "noise"]).shape[0],
         majority_rate=sfb.majority_baseline(D3all.label)),
    dict(population="hydrophone recordings (all D3 accuracy results)", n=len(H),
         composition="H1=60; H2=60", n_runs=H.run_id.nunique(),
         majority_rate=sfb.majority_baseline(H.label)),
    dict(population="hydrophone 1 s segments (E5)", n=len(H) * 30,
         composition="30 segments per recording", n_runs=H.run_id.nunique(),
         majority_rate=sfb.majority_baseline(H.label)),
])
mapping.to_csv(cfg.RESULTS / "D3_population_mapping.csv", index=False)
print(mapping.to_string(index=False))

tables = []
for pop_name, d in [("all280", D3all), ("hydro120", H)]:
    for cols in [["topology"], ["flow"], ["noise"], ["sensor"], ["modality"],
                 ["topology", "flow"], ["flow", "noise"], ["topology", "flow", "noise"],
                 ["topology", "flow", "noise", "sensor"]]:
        t = d.groupby(cols + ["label"]).size().unstack(fill_value=0).reset_index()
        t["p_leak"] = t["leak"] / (t["leak"] + t["noleak"])
        t.insert(0, "factors", "x".join(cols)); t.insert(0, "population", pop_name)
        tables.append(t)
cbc = pd.concat(tables, ignore_index=True)
cbc.to_csv(cfg.RESULTS / "D3_class_by_condition.csv", index=False)
print("\nP(leak) in every cell (should be 0.8 throughout for a balanced factorial):")
print(cbc.groupby(["population", "factors"]).p_leak.agg(["min", "max", "size"]).to_string())
print("\nexample: hydrophone, flow x noise")
print(cbc[(cbc.population == "hydro120") & (cbc.factors == "flowxnoise")].to_string(index=False))

                                     population    n                           composition  n_runs  majority_rate
                all modalities (v1 Table 3 SFB)  280 hydrophone=120; accel=80; pressure=80     100            0.8
hydrophone recordings (all D3 accuracy results)  120                          H1=60; H2=60      60            0.8
                   hydrophone 1 s segments (E5) 3600             30 segments per recording      60            0.8

P(leak) in every cell (should be 0.8 throughout for a balanced factorial):
                                       min  max  size
population factors                                   
all280     flow                        0.8  0.8     4
           flowxnoise                  0.8  0.8    10
           modality                    0.8  0.8     3
           noise                       0.8  0.8     3
           sensor                      0.8  0.8     6
           topology                    0.8  0.8     2
           topologyxflow            

## 2. Table 1 (D3)

In [3]:
t1 = pd.DataFrame([dict(dataset="D3", task="leak vs no-leak (CC/GL/LC/OL vs NL)",
                        unit="hydrophone recording (accuracy: 1 s segments)",
                        n=len(H), class_counts="; ".join(f"{k}={v}" for k, v in H.label.value_counts().items()),
                        majority_rate=sfb.majority_baseline(H.label),
                        groups=f"{H.run_id.nunique()} runs, 6 operating conditions, 2 topologies")])
t1.to_csv(cfg.RESULTS / "table1_D3.csv", index=False)
print(t1.to_string(index=False))

dataset                                task                                          unit   n       class_counts  majority_rate                                        groups
     D3 leak vs no-leak (CC/GL/LC/OL vs NL) hydrophone recording (accuracy: 1 s segments) 120 leak=96; noleak=24            0.8 60 runs, 6 operating conditions, 2 topologies


## 3. Feature extraction by window mode

- Modes: min-std, random, first, full
- Band-pass filter → window selection → summary features (199)
- Cached with file IDs

In [4]:
MODALITY = "hydrophone"

def build_features(mode, sec=cfg.WIN_SEC, seed=0, decoding="pcm32"):
    name = f"d3_{MODALITY}_{mode}_{sec}_{decoding}"
    c = io_utils.load_cache(name, H.file_id)
    if c is not None:
        return c["X"]
    Xs = []
    for i, r in tqdm(H.iterrows(), total=len(H), desc=f"{MODALITY}/{mode}"):
        x, sr = io_utils.load_d3_signal(r.to_dict(), decoding=decoding)
        xb = features.bandpass(x, sr)
        xw = features.pick_window(xb, sr, sec=sec, mode=mode, seed=seed + i)
        Xs.append(features.summary_features(xw, sr))
    X = np.vstack(Xs)
    io_utils.save_cache(name, X, H.file_id)
    return X

y3 = H.label.to_numpy()
X3 = {m: build_features(m) for m in ["minstd", "random", "first", "full"]}
for m, Xm in X3.items():
    print(f"  {m:8s} -> {Xm.shape}")

hydrophone/minstd:   0%|          | 0/120 [00:00<?, ?it/s]

hydrophone/random:   0%|          | 0/120 [00:00<?, ?it/s]

hydrophone/first:   0%|          | 0/120 [00:00<?, ?it/s]

hydrophone/full:   0%|          | 0/120 [00:00<?, ?it/s]

  minstd   -> (120, 199)
  random   -> (120, 199)
  first    -> (120, 199)
  full     -> (120, 199)


## 4. E8: window selection under three split designs

Model: GBDT. Same splits for all window modes.

- `naive`: stratified 70/30 split of recordings, 10 seeds
- `condition_indep_v1`: v1 grouping (topology × leak type), 70/30; draws without both classes excluded (all draws logged)
- `opcond_LOCO`: leave-one-operating-condition-out, 6 folds, OOF over 120 recordings, 10 seeds

In [5]:
rows, attempts = [], []
cond_v1 = H.cond_topo_leak.to_numpy()
for seed in range(cfg.N_SEEDS):
    a_n, b_n = train_test_split(np.arange(len(y3)), test_size=.3, random_state=seed, stratify=y3)
    a_c, b_c = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                    .split(np.arange(len(y3)), y3, groups=cond_v1))
    usable = len(np.unique(y3[a_c])) > 1 and len(np.unique(y3[b_c])) > 1
    attempts.append(dict(seed=seed, held_out=",".join(sorted(set(cond_v1[b_c]))), usable=usable,
                         reason="" if usable else ("no no-leak recording in test" if len(np.unique(y3[b_c])) < 2
                                                   else "no no-leak recording in train")))
    for mode, Xm in X3.items():
        rows.append(dict(mode=mode, split="naive", seed=seed,
                         **{k: v for k, v in ev.evaluate(Xm[a_n], y3[a_n], Xm[b_n], y3[b_n], "gbdt", seed).items()
                            if k in ("acc", "bacc", "base", "excess")}))
        if usable:
            rows.append(dict(mode=mode, split="condition_indep_v1", seed=seed,
                             **{k: v for k, v in ev.evaluate(Xm[a_c], y3[a_c], Xm[b_c], y3[b_c], "gbdt", seed).items()
                                if k in ("acc", "bacc", "base", "excess")}))
loco = ev.make_folds("logo", y3, H.op_condition.to_numpy())
oof_ok = {}
for mode, Xm in X3.items():
    for seed in range(cfg.N_SEEDS):
        pred, _ = ev.oof_predict(Xm, y3, loco, "gbdt", seed)
        s = ev.score(y3, pred)
        rows.append(dict(mode=mode, split="opcond_LOCO", seed=seed, acc=s["acc"], bacc=s["bacc"],
                         base=s["base"], excess=s["excess"]))
        if seed == 0:
            oof_ok[mode] = pred == y3

e8 = pd.DataFrame(rows)
pd.DataFrame(attempts).to_csv(cfg.RESULTS / "E8_condition_split_attempts.csv", index=False)
s8 = e8.groupby(["mode", "split"]).agg(acc=("acc", "mean"), sd=("acc", "std"), bacc=("bacc", "mean"),
                                       base=("base", "mean"), n=("acc", "size")).reset_index()
print(s8.round(4).to_string(index=False))
e8.to_csv(cfg.RESULTS / "E8_window_heuristic_raw.csv", index=False)
s8.to_csv(cfg.RESULTS / "E8_window_heuristic_summary.csv", index=False)
print("\nv1 condition split attempts:")
print(pd.DataFrame(attempts).to_string(index=False))

  mode              split    acc     sd   bacc   base  n
 first condition_indep_v1 0.6222 0.0421 0.4667 0.6667  5
 first              naive 0.8028 0.0443 0.5687 0.8056 10
 first        opcond_LOCO 0.8333 0.0000 0.6302 0.8000 10
  full condition_indep_v1 0.6722 0.0124 0.5375 0.6667  5
  full              naive 0.7944 0.0438 0.5690 0.8056 10
  full        opcond_LOCO 0.8250 0.0000 0.5938 0.8000 10
minstd condition_indep_v1 0.6167 0.0232 0.4625 0.6667  5
minstd              naive 0.8111 0.0388 0.6172 0.8056 10
minstd        opcond_LOCO 0.8583 0.0000 0.7083 0.8000 10
random condition_indep_v1 0.6333 0.0602 0.4833 0.6667  5
random              naive 0.8028 0.0381 0.5687 0.8056 10
random        opcond_LOCO 0.8333 0.0000 0.6302 0.8000 10

v1 condition split attempts:
 seed          held_out  usable                        reason
    0 BR_LC,BR_OL,LO_NL    True                              
    1 BR_LC,LO_GL,LO_OL   False  no no-leak recording in test
    2 BR_GL,BR_OL,LO_CC   False  no no-leak

### 4.1 min-std vs. random window (opcond_LOCO, seed 0)

Paired bootstrap of the difference, resampling physical runs

In [6]:
cmp = sfb.paired_cluster_bootstrap(oof_ok["minstd"], oof_ok["random"], H.run_id, n_boot=cfg.N_BOOT)
print("opcond_LOCO, seed 0, minstd - random:", {k: round(v, 4) if isinstance(v, float) else v
                                                for k, v in cmp.items()})
la.save_json(cmp, "E8_minstd_vs_random_paired_runs.json")
rp.record("D3.E8.minstd_minus_random.opcond_LOCO", cmp["diff"], "E8_minstd_vs_random_paired_runs.json",
          "self-audit", lo=cmp["lo"], hi=cmp["hi"], note="unit=physical run")

opcond_LOCO, seed 0, minstd - random: {'diff': 0.025, 'lo': -0.0417, 'hi': 0.0917, 'excludes_zero': False, 'p_boot': 0.424, 'n_units': 120, 'n_clusters': 60}
